# Annotated dataset volumes and ventricle symmetry

Select a normalized **Dx folder**, then inspect all subjects and timepoints. The report measures original annotation IDs on each native physical grid, retaining missing annotations, true zeros and negative corrected estimates. No model is trained or evaluated.

The correction is **lesion + ipsilateral ventricle - contralateral ventricle**, without clipping. SHAM lesion absence is known. Ipsilateral/contralateral meanings follow the existing surgery/lesion convention. Longitudinal scans are averaged within animals before condition-specific means and SDs. The ratio is averaged per scan within each animal; zero contralateral volume has an undefined ratio.

Volume measurement does not require reorientation. The shared RIA reference is retained for any image inspection; the report does not resample annotations.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import display
import plotly.io as pio

pio.renderers.default = "vscode"
REFERENCE: Path = REPO_ROOT / "example/RARE/TBI_fm_19_49/Anat/TBI_fm_19_49_N4.nii.gz"


In [ ]:
from src.evaluation.volumes import VolumeLabels, dataset_volume_report, ventricle_ratio_figure

DATASET_DIR: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized/D3")
LABELS = VolumeLabels(lesion=1, ipsilateral=13, contralateral=3)
SPATIAL_UNIT: str | None = None  # Read NIfTI units; override only when known, e.g. "mm".
RUN_REPORT: bool = False
EXPORT_REPORT: bool = False
OUTPUT_DIR: Path = REPO_ROOT / "outputs/dataset_volumes" / DATASET_DIR.name


In [ ]:
report = None
if RUN_REPORT:
    report = dataset_volume_report(DATASET_DIR, labels=LABELS, spatial_unit=SPATIAL_UNIT)
    display(report.scans)
    display(report.animals)
    display(report.summary)
else:
    print("Select a Dx folder, then set RUN_REPORT=True.")


## SHAM symmetry distribution

Use the SHAM rows for the healthy ventricle check. Mean ratio near one is descriptive evidence, not proof of symmetry for every animal. Inspect SD, range, missing/zero-denominator scans and negative corrected-volume values. TBI ratios remain separately reported.

In [ ]:
if report is not None:
    sham_ratios = report.summary.loc[
        report.summary["Condition"].eq("SHAM")
        & report.summary["Measurement"].eq("IpsiContraRatio")
    ]
    display(sham_ratios)
    figure = ventricle_ratio_figure(report)
    figure.show(renderer="vscode")


## Export

CSV tables and an Excel workbook go to a fresh folder. The original annotation files remain unchanged. Corrected and uncorrected volumes are exported for later analysis; this notebook does not run a D4 agreement or interpolation sensitivity study.

In [ ]:
if EXPORT_REPORT:
    if report is None:
        raise ValueError("Run the dataset report before exporting")
    saved_dir = report.export(OUTPUT_DIR)
    print("Saved to:", saved_dir)
